# 01_user_listening_summary

DML: gold_user_listening_summary — Aggregate: total tracks, artists, minutes, favorites.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [ ]:
from pyspark.sql import functions as F

fct   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
dim_t = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_tracks").select("track_id", "duration_ms")
bph   = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_play_history").select("played_at", "track_id", "artist_ids")

unique_artists_count = (
    fct.join(bph, ["played_at", "track_id"], "left")
    .select(F.explode("artist_ids").alias("artist_id"))
    .agg(F.countDistinct("artist_id").alias("unique_artists"))
)

summary = (
    fct.join(dim_t, "track_id", "left")
    .agg(
        F.count("play_id").alias("total_plays"),
        F.countDistinct("track_id").alias("unique_tracks"),
        F.round(F.sum(F.coalesce(F.col("duration_ms"), F.lit(0)) / 60000.0), 2).alias("total_minutes"),
    )
    .crossJoin(unique_artists_count)
    .withColumn("snapshot_date", F.to_date(F.lit(snapshot_date)))
    .select("snapshot_date", "total_plays", "unique_tracks", "unique_artists", "total_minutes")
)

upsert_delta(summary, f"{CATALOG}.{GOLD_SCHEMA}.gold_user_listening_summary", ["snapshot_date"])
display(summary)